In [ ]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler
# Initialize Langfuse client
langfuse = get_client()
# Initialize Langfuse CallbackHandler for Langchain (tracing)
langfuse_handler = CallbackHandler()

In [ ]:
# Core LLM + tool + graph imports
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
# In-memory checkpoint store for multi-turn thread conversations
from langgraph.checkpoint.memory import MemorySaver


# Standard library typing/date utilities
from datetime import datetime
from typing import List, Optional, Annotated, Union, Tuple
from typing_extensions import TypedDict
from pydentic import BaseModel, Field


In [ ]:
# Initialize the base chat model and attach tracing callbacks
LLM = ChatNVIDIA(
    model="openai/gpt-oss-120b",
    # model="meta/llama-3.1-70b-instruct",
    temperature=0,
    callbacks=[langfuse_handler],
)

In [ ]:
search_tool = GoogleSerperAPIWrapper()

@tool
def search(query: str) -> str:
    """Search the internet for information about a query and return the results.
    """
    return search_tool.run(query)

In [ ]:
class State(TypedDict):
    input : str
    plan : List[str]
    past_steps : Annotated[list[tuple[str, str]], operator.add] # List of (tool_calls, tool_results) tuples